# Chapter 11 &mdash; The Complement of a Non-CFL Can Be a CFL

**Concept 20 of the Chapter 11 decomposition:** *The Complement of a Non-CFL Can Be a CFL: $\overline{L_{ww}}$*

$\overline{L_{ww}}$ <i>is</i> context-free &mdash; characterise its members by a mismatching position.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter11/Concept-Complement-Of-Lww/Concept-Complement-Of-Lww.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --

#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$L_{ww}$ is not context-free (Concept 19), yet **its complement is**. That is only
possible because CFLs are not closed under complement &mdash; and it is the cleanest
illustration of that fact.

The characterisation. A string is **not** of the form $ww$ if either

* it has **odd length** &mdash; easy; or
* it has **even length** and there is a position $p$ where the two halves **differ**.

The second case is the interesting one, and the trick (Concept 21) is to re-view it so
that a grammar can generate it: instead of "position $p$ in the first half differs
from position $p$ in the second half", write the string as
$p\,0\,q\ \ p'\,1\,q'$ with $|p|=|p'|$ and $|q|=|q'|$, and then **rearrange**.

## 2. Definitions

### The CFG toolkit

A grammar is a dict; `language`, `nparses`, `parse_trees` and `leftmost` do the work.

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps

### The two cases, as predicates

In [ ]:
def in_ww(s):
    return len(s) % 2 == 0 and s[:len(s)//2] == s[len(s)//2:]

def odd_len(s): return len(s) % 2 == 1

def even_mismatch(s):
    if len(s) % 2: return False
    h = len(s)//2
    return any(s[i] != s[h+i] for i in range(h))

### The grammar (Concept 21 derives it)

In [ ]:
NotWW = mkg({'S':  ["A", "B", "AB", "BA"],
             'A':  ["a", "aAa", "aAb", "bAa", "bAb"],   # odd, centre 'a'
             'B':  ["b", "aBa", "aBb", "bBa", "bBb"]})  # odd, centre 'b'

## 3. Tests

The two cases are exhaustive and disjoint.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(9) for p in product('ab', repeat=k)]
for s in strs:
    assert (not in_ww(s)) == (odd_len(s) or even_mismatch(s)), s
    assert not (odd_len(s) and even_mismatch(s))
print("not-ww  ==  odd length OR even with a mismatching position, on all %d strings"
      % len(strs))

The grammar generates exactly the complement.

In [ ]:
L = set(language(NotWW, 7))
want = {s for s in strs if len(s) <= 7 and not in_ww(s)}
print("generated %d, intended %d" % (len(L), len(want)))
print("missing :", sorted(want - L)[:6])
print("extra   :", sorted(L - want)[:6])
assert L == want

**So the complement of a non-CFL is a CFL.** That settles non-closure concretely.

In [ ]:
print("L_ww          : NOT context-free   (Concept 19)")
print("complement    : context-free       (this grammar)")
print()
print("If CFLs were closed under complement, L_ww would be context-free too.")
print("They are not.  Contrast Chapter 6: regular languages ARE closed.")

Sanity: the odd-length members are all there.

In [ ]:
odd = sorted(w for w in L if len(w) % 2)
print("odd-length members up to 5 :", [w for w in odd if len(w) <= 3])
assert all(odd_len(w) or even_mismatch(w) for w in L)
assert 'ab' in L and 'abab' not in L
print("\n'ab' is not ww (a != b); 'abab' IS ww with w = 'ab', so it is excluded.")

Counting, as a cross-check.

In [ ]:
for n in range(0, 8):
    tot = 2 ** n
    ww  = 2 ** (n // 2) if n % 2 == 0 else 0
    got = len([w for w in L if len(w) == n])
    print("  length %d : 2^%d = %3d strings, %3d are ww, complement %3d (grammar %3d)"
          % (n, n, tot, ww, tot - ww, got))
    assert got == tot - ww

## 4. Exercises


1. Why is "odd length" a CFL on its own? Is it even regular?
2. Check that `A` generates exactly the odd-length strings with `a` in the centre.
3. Why do `AB` and `BA` both appear in the start rule?

In [ ]:
# Your work for the exercises above.